In [ ]:
FILE_IN = 'mia.txt'     # File name in Downloads folder

################################################################################
import os, time, re, json, subprocess, sys
import importlib.util as il

if None in [il.find_spec('python-ulid'), il.find_spec('pyperclip'), il.find_spec('pandas')]:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-ulid']);
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyperclip']);
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas']);
    
from ulid import ULID
import pyperclip
import pandas as pd

def gen_ulid():
    return str(ULID.from_timestamp(time.time()))

def convert_coord(c):
    c = str(c)
    j = len(c) - 6
    d = int(c[0:2 + j])
    m = int(c[2 + j:4 + j])
    s = float(c[4 + j:6 + j] + '.' + c[6 + j:])
    q = 1 if j == 0 else -1
    coord = round(q * (d + m / 60 + s / 3600), 6)
    
    return coord

def pprint(dict):
    print(json.dumps(dict, indent=2))

def comma_followed_by_number(s):
    for i, char in enumerate(s[:-1]):
        if char == ',' and s[i+1].isdigit():
            return True
    return False

def extract_table_section_from_file(section_header, filename, offset=0):
    offset *= 3
    section_header = '******* ' + section_header + ' *******'

    downloads_folder = os.path.join(os.path.expanduser("~"), "Downloads")
    with open(os.path.join(downloads_folder, filename), "r") as file:
        lines = file.readlines()

    extracted_lines = []
    inside_section = False
    end_marker_count = 0

    for line in lines:
        if section_header in line:
            inside_section = True
            extracted_lines.append(line)
            continue

        if inside_section:
            if end_marker_count > offset:
                extracted_lines.append(line)
            # Count lines that are mostly dashes
            if line.strip().startswith('---'):
                end_marker_count += 1
                if end_marker_count >= 3 + offset:
                    break

    return "".join(extracted_lines)

def remove_dash_lines(text):
    cleaned_lines = [
        line for line in text.splitlines()
        if not line.strip().startswith("---")
    ]
    return "\n".join(cleaned_lines)

def convert_pipe_text_to_csv(multi_line_text):
    csv_lines = []
    for line in multi_line_text.splitlines():
        if not line.strip():
            continue
        if '|' not in line:
            continue
        
        fields = [field.strip() for field in line.strip('|').split('|')]
        csv_line = '|'.join(fields)
        csv_lines.append(csv_line)

    return '\n'.join(csv_lines)

def csv_text_to_dataframe(csv_text):
    lines = [line.strip() for line in csv_text.strip().split('\n') if line.strip()]
    
    headers = [h.strip() for h in lines[0].split('|')]
    
    data = []
    for line in lines[1:]:
        fields = [f.strip() for f in line.split('|')]
        data.append(fields)
    
    df = pd.DataFrame(data, columns=headers)
    return df

def read_adaptation_section(section_header, filename, offset=0):
    text = extract_table_section_from_file(section_header, filename, offset)
    text = remove_dash_lines(text)
    text = convert_pipe_text_to_csv(text)
    
    return csv_text_to_dataframe(text)

In [ ]:
filename = 'mia.txt' # FILE_IN
USE_NUM = False
facility = filename.replace('.txt', '').upper()
maps = read_adaptation_section('MAP', filename).drop(columns=['#.', 'Predefined Objects', 'Origin Lat', 'Origin N/S'])
column_order = ['Map Number', 'map_group', 'Mnemonic', 'Map Title']
maps = maps[column_order]

base_path = os.path.join(os.path.expanduser("~"), "Downloads")
maps['File'] = ''

for index, row in maps.iterrows():
    map_num = str(row['Map Number']).strip().zfill(3)

    geojson_name = ''
    if USE_NUM:
        geojson_name = f"{facility}-{map_num}S.geojson"
    else:
        if 'RVM' in row['Map Title']:
            geojson_name = f"{row['Map Title'][3:6]}-{row['Map Title'][6:]}.geojson"
        else:
            pass
    full_path = os.path.join(base_path, facility, geojson_name)
    
    if os.path.exists(full_path):
        maps.at[index, 'File'] = geojson_name
    elif os.path.exists(full_path.replace('MIA', 'FLL')):
        maps.at[index, 'File'] = geojson_name
    else:
        pass

maps2 = read_adaptation_section('MAP', filename, offset=1)
maps['Bright Cat'] = maps2['Map Bright Cat']

maps.to_csv(os.path.join(os.path.expanduser("~"), "Downloads") + os.sep + facility + '_map_data.csv', sep=',', index=False)

In [ ]:
def reconcile_map_files(facility):
    csv_path = r"C:\Users\Josh\Downloads" + os.sep + facility + "_map_data.csv"
    base_dir = os.path.join(os.path.expanduser("~"), "Downloads", "maps_geojson", "cleaned", facility)
    
    if not os.path.exists(csv_path):
        return f"Error: {csv_path} not found."
    
    df = pd.read_csv(csv_path)
    files_in_csv = set(df['File'].dropna().unique())

    if not os.path.exists(base_dir):
        return f"Error: Directory {base_dir} does not exist."
    
    all_physical_files = [f for f in os.listdir(base_dir) if f.endswith('.geojson')]

    report = []
    for filename in all_physical_files:
        is_included = filename in files_in_csv
        report.append({
            'In_CSV': 'Yes' if is_included else 'No',
            'Filename': filename
        })

    return pd.DataFrame(report)

# Example Usage:
result_df = reconcile_map_files(filename.replace('.txt', '').upper())
print(result_df.to_csv(index=False))

In [ ]:
# Define paths
downloads_path = os.path.join(os.path.expanduser("~"), "Downloads")
csv_path = os.path.join(downloads_path, "map_data.csv")
json_path = os.path.join(downloads_path, "videomaps.json")
output_path = os.path.join(downloads_path, "videomaps_updated.json")

# 1. Load the CSV
# We filter out rows where 'File' is empty
df = pd.read_csv(csv_path)
df = df.dropna(subset=['File'])

# 2. Create Lookup Dictionary from CSV
# Key = File
# Values = Name, Short Name, Bright Cat, Map Number
lookup = df.set_index('File')[['Name', 'Short Name', 'Bright Cat', 'Map Number']].to_dict('index')

# 3. Load the JSON
with open(json_path, 'r') as f:
    data = json.load(f)

# 4. Iterate through JSON and update matching entries
if 'videoMaps' in data:
    for entry in data['videoMaps']:
        source_file = entry.get('sourceFileName')
        
        # If the filename from JSON exists in our CSV lookup table
        if source_file in lookup:
            match = lookup[source_file]
            
            # Name -> name
            if pd.notna(match['Name']):
                entry['name'] = str(match['Name'])
                
            # Short Name -> shortName
            if pd.notna(match['Short Name']):
                entry['shortName'] = str(match['Short Name'])
                
            # Bright Cat -> starsBrightnessCategory
            if pd.notna(match['Bright Cat']):
                entry['starsBrightnessCategory'] = str(match['Bright Cat'])
                
            # Map Number -> starsId
            if pd.notna(match['Map Number']):
                entry['starsId'] = int(match['Map Number'])

# 5. Save the updated JSON
with open(output_path, 'w') as f:
    json.dump(data, f, indent=4)

print(f"Successfully created {output_path}")